# Clipper Engine - Kaggle Setup (Git Version)
This notebook clones the Clipper Engine from GitHub, installs it on a Kaggle T4 GPU, and exposes the web UI using Ngrok.

**Security Note:** Never upload your API keys or tokens directly to GitHub. This notebook uses Kaggle Secrets to securely load your Ngrok token.

In [ ]:
# 1. Install System Dependencies (FFmpeg)
!apt-get update
!apt-get install -y ffmpeg

In [ ]:
# 2. Clone Codebase from GitHub

import os

REPO_URL = "https://github.com/advait2004/clipper-engine.git"

# Remove old clone if it exists (useful for rerunning the cell)
!rm -rf /kaggle/working/clipper_engine_repo

# Clone the repository
!git clone {REPO_URL} /kaggle/working/clipper_engine_repo

# Navigate into the 'clipper' directory
os.chdir("/kaggle/working/clipper_engine_repo/clipper")
print("\nCurrent directory:", os.getcwd())

In [ ]:
# 3. Install Python Dependencies
!pip install -r requirements.txt
!pip install pyngrok

In [ ]:
# 4. Expose Web UI via Ngrok securely
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
import os

# ---------------------------------------------------------
# ⚠️ IMPORTANT: In your Kaggle Notebook, go to the top menu:
# Add-ons -> Secrets.
# Add a new secret with Label: NGROK_AUTH_TOKEN and Value: your_ngrok_token
# (You can also add GROQ_API_KEY or GEMINI_API_KEY the same way!)
# ---------------------------------------------------------

user_secrets = UserSecretsClient()

try:
    ngrok_token = user_secrets.get_secret("NGROK_AUTH_TOKEN")
    ngrok.set_auth_token(ngrok_token)
except Exception as e:
    print("❌ Could not find NGROK_AUTH_TOKEN in Kaggle Secrets! Please add it.")
    raise e

# Optional: load API keys securely so they aren't exposed in GitHub
try:
    os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")
    print("Loaded GROQ_API_KEY securely from Kaggle Secrets")
except:
    pass

try:
    os.environ["GEMINI_API_KEY"] = user_secrets.get_secret("GEMINI_API_KEY")
    print("Loaded GEMINI_API_KEY securely from Kaggle Secrets")
except:
    pass

public_url = ngrok.connect(8000)

print(f"\n\n=========================================")
print(f"\n🚀 CLIPPER ENGINE UI RUNNING AT:\n   {public_url}\n")
print(f"=========================================\n\n")

In [ ]:
# 5. Run the FastAPI Server
!python -m uvicorn app:app --host 127.0.0.1 --port 8000